In [1]:
from pathlib import Path
import time
import requests
import pandas as pd 

In [2]:
RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True) 

In [3]:
URL_ECO2MIX = ("https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
               "eco2mix-national-cons-def/exports/csv")
params = {"select": "date_heure,consommation", "delimiter": ",", "order_by": "date_heure"}

r = requests.get(URL_ECO2MIX, params=params, timeout=300)
r.raise_for_status()
(RAW / "eco2mix.csv").write_bytes(r.content)

15685324

In [4]:
URL_ECO2MIX = ("https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
               "eco2mix-national-cons-def/exports/csv")
params = {"select": "date_heure,consommation", "delimiter": ",", "order_by": "date_heure"}

r = requests.get(URL_ECO2MIX, params=params, timeout=300)
r.raise_for_status()
(RAW / "eco2mix.csv").write_bytes(r.content)

15685324

In [5]:
conso = pd.read_csv(RAW / "eco2mix.csv", encoding="utf-8-sig")
print(conso.shape)
conso.head(10)

(514272, 2)


,date_heure,consommation
0,2011-12-31T23:00:00+00:00,58315.0
1,2011-12-31T23:15:00+00:00,NaN
2,2011-12-31T23:30:00+00:00,58315.0
3,2011-12-31T23:45:00+00:00,NaN
4,2012-01-01T00:00:00+00:00,56231.0
5,2012-01-01T00:15:00+00:00,NaN
6,2012-01-01T00:30:00+00:00,56075.0
7,2012-01-01T00:45:00+00:00,NaN
8,2012-01-01T01:00:00+00:00,55532.0
9,2012-01-01T01:15:00+00:00,NaN


In [ ]:
VILLES = {
    "Paris": (48.8566, 2.3522), "Lyon": (45.7640, 4.8357), "Marseille": (43.2965, 5.3698), "Toulouse": (43.6047, 1.4442),
    "Nice": (43.7102, 7.2620), "Nantes": (47.2184, -1.5536), "Strasbourg": (48.5734, 7.7521), "Montpellier": (43.6119, 3.8772),
    "Bordeaux": (44.8378, -0.5792), "Lille": (50.6292, 3.0573), "Rennes": (48.1173, -1.6778), "Reims": (49.2583, 4.0317), 
    "Le Havre": (49.4944, 0.1079), "Saint-Étienne": (45.4397, 4.3872), "Toulon": (43.1242, 5.9280), "Grenoble": (45.1885, 5.7245),
    "Dijon": (47.3220, 5.0415), "Angers": (47.4784, -0.5632), "Nîmes": (43.8367, 4.3601), "Villeurbanne": (45.7667, 4.8833), "Clermont-Ferrand": (45.7772, 3.0870),
}
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"

frames = []
for ville, (lat, lon) in VILLES.items():
    r = requests.get(URL_METEO, params={
        "latitude": lat,
        "longitude": lon,
        "start_date": "2012-01-01",
        "end_date": "2026-08-31",
        "hourly": "temperature_2m",
        "timezone" : "UTC",
    }, timeout=300)
    r.raise_for_status()
    h = r.json()["hourly"]
    frames.append(pd.DataFrame({"time": h["time"], "temperature_2m": h["temperature_2m"], "ville": ville}))
    print(f"{ville} OK")
    time.sleep(1)

    meteo = pd.concat(frames, axis=1)
    meteo.to_csv(RAW / "meteo.csv")
    meteo.head()